In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

In [ ]:
%%writefile /kaggle/working/rewind_agent.py
"""
RewindAgent v15 — BFS + Online Hybrid
When game source is available (Kaggle), use offline BFS to find solutions.
When online-only (API), use frame-based exploration.
"""

import copy
import hashlib
import importlib.util
import logging
import os
import time
from collections import deque
from pathlib import Path
from typing import Any, Optional

import numpy as np

from arcengine import ActionInput, FrameData, GameAction, GameState
from agents.agent import Agent

logger = logging.getLogger(__name__)


class BFSSolver:
    """Offline BFS solver using local game source."""

    def __init__(self, game_path, class_name, scan_timeout=3, bfs_timeout=90):
        self.game_path = game_path
        self.class_name = class_name
        self.scan_timeout = scan_timeout
        self.bfs_timeout = bfs_timeout
        self.game_cls = None
        self.solutions = {}  # level_idx -> action list

    def load(self):
        try:
            spec = importlib.util.spec_from_file_location('game_mod', self.game_path)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            self.game_cls = getattr(mod, self.class_name)
            return True
        except Exception as e:
            logger.warning(f'BFS: Failed to load {self.class_name}: {e}')
            return False

    def _state_hash(self, frame):
        return hashlib.md5(frame.tobytes()).hexdigest()[:16]

    def _scan_actions(self, game, f0, bg):
        avail = game._available_actions
        actions = []
        for a in [x for x in avail if x <= 5]:
            g = copy.deepcopy(game)
            try:
                r = g.perform_action(ActionInput(id=GameAction.from_id(a)), raw=True)
                if r.frame and np.sum(f0 != np.array(r.frame[-1])) > 0:
                    actions.append((a, None))
            except: pass
        if 6 in avail:
            t0 = time.time()
            seen = set()
            for y in range(0, 64, 2):
                if time.time() - t0 > self.scan_timeout: break
                for x in range(0, 64, 2):
                    if f0[y, x] == bg: continue
                    g = copy.deepcopy(game)
                    try:
                        r = g.perform_action(ActionInput(id=GameAction.ACTION6, data={'x': x, 'y': y, 'game_id': 'bfs'}), raw=True)
                        if not r.frame: continue
                        f = np.array(r.frame[-1])
                        diff = np.sum(f0 != f)
                        if diff > 0:
                            eh = hashlib.md5(f.tobytes()).hexdigest()[:12]
                            if eh not in seen:
                                seen.add(eh)
                                actions.append((6, {'x': x, 'y': y, 'game_id': 'bfs'}))
                    except: pass
        return actions

    def solve_level(self, level_idx, max_states=200000, prev_solution=None):
        if not self.game_cls: return None
        game = self.game_cls()
        game.set_level(level_idx)
        game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
        r0 = game.perform_action(ActionInput(id=GameAction.RESET), raw=True)
        if not r0.frame: return None
        f0 = np.array(r0.frame[-1])
        bg = int(np.bincount(f0.flatten(), minlength=16).argmax())

        # Transfer from previous level
        if prev_solution and level_idx > 0:
            g = copy.deepcopy(game)
            try:
                for act_id, data in prev_solution:
                    ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                    r = g.perform_action(ai, raw=True)
                if r.state == GameState.NOT_FINISHED and r.levels_completed > level_idx - 1:
                    return prev_solution
            except: pass

        actions = self._scan_actions(game, f0, bg)
        if not actions:
            logger.info(f'BFS L{level_idx}: no effective actions found')
            return None

        logger.info(f'BFS L{level_idx}: {len(actions)} unique actions, searching...')
        t0 = time.time()
        visited = {self._state_hash(f0)}
        queue = deque([(game, [])])  # (game_state, action_sequence)
        best = None
        states = 0

        while queue and states < max_states and (time.time() - t0) < self.bfs_timeout:
            g_state, path = queue.popleft()
            states += 1

            for act_id, data in actions:
                g = copy.deepcopy(g_state)
                try:
                    ai = ActionInput(id=GameAction.from_id(act_id), data=data) if data else ActionInput(id=GameAction.from_id(act_id))
                    r = g.perform_action(ai, raw=True)
                except: continue

                if not r.frame: continue
                f = np.array(r.frame[-1])
                h = self._state_hash(f)

                if r.levels_completed > level_idx - 1 + (1 if level_idx > 0 else 0):
                    # SOLVED!
                    solution = path + [(act_id, data)]
                    logger.info(f'BFS L{level_idx}: SOLVED in {len(solution)} actions, {states} states, {time.time()-t0:.1f}s')
                    return solution

                if r.state == GameState.GAME_OVER: continue
                if h in visited: continue
                visited.add(h)
                queue.append((g, path + [(act_id, data)]))

        logger.info(f'BFS L{level_idx}: no solution ({states} states, {time.time()-t0:.1f}s)')
        return None


class RewindAgent(Agent):
    MAX_ACTIONS: int = 500

    def __init__(self, *a, **kw):
        super().__init__(*a, **kw)
        self.levels = 0
        self.queue = []
        self.phase = 'INIT'
        self.attempt = 0
        self.game_type = None
        self.available = None
        self.door_pat = None
        self.bfs_solver = None
        self.bfs_solutions = {}  # level -> [(action_id, data)]
        self.current_solution_idx = 0
        self._try_load_bfs()
        logger.info('RewindAgent v15 init')

    def _try_load_bfs(self):
        """Try to load game source for BFS (only works on Kaggle)."""
        env_dir = os.environ.get('ENVIRONMENTS_DIR', 'environment_files')
        if not env_dir: return
        game_short = self.game_id.split('-')[0]
        class_name = game_short[0].upper() + game_short[1:]

        candidates = [
            Path(env_dir) / game_short / f'{game_short}.py',
            Path(env_dir) / game_short / f'{class_name}.py',
            Path(env_dir) / f'{game_short}.py',
        ]
        for p in candidates:
            if p.exists():
                self.bfs_solver = BFSSolver(str(p), class_name)
                if self.bfs_solver.load():
                    logger.info(f'BFS solver loaded from {p}')
                    # Pre-solve levels
                    prev = None
                    for lv in range(7):
                        sol = self.bfs_solver.solve_level(lv, prev_solution=prev)
                        if sol:
                            self.bfs_solutions[lv] = sol
                            prev = sol
                        else:
                            break
                    logger.info(f'BFS pre-solved {len(self.bfs_solutions)}/7 levels')
                    return
                else:
                    self.bfs_solver = None

    # ===== Online fallback methods (same as v14) =====
    def _get_arr(self, frame):
        return np.array(frame.frame[0]) if frame.frame else None

    def _detect_type(self, frame):
        aa = frame.available_actions or []
        has_click = 6 in aa
        has_kbd = any(a in aa for a in [1, 2, 3, 4])
        if has_click and not has_kbd: return 'click'
        elif has_kbd and not has_click: return 'keyboard'
        elif has_click and has_kbd: return 'mixed'
        else: return 'keyboard'

    def _find_player(self, arr):
        p = np.where(arr[:52] == 12)
        return (int(round(p[0].mean())), int(round(p[1].mean()))) if len(p[0]) > 0 else None

    def _get_key(self, arr):
        try:
            return tuple(tuple(1 if int(arr[55+r*2,3+c*2])==9 else 0 for c in range(3)) for r in range(3))
        except: return None

    def _find_rotator(self, arr):
        g = arr[:52]; rs, cs = [], []
        for v in [0,1]:
            l = np.where(g==v); rs.extend(l[0].tolist()); cs.extend(l[1].tolist())
        return (int(round(np.mean(rs))), int(round(np.mean(cs)))) if rs else None

    def _find_door(self, arr):
        g = arr[:52]; nines = np.where(g==9)
        if len(nines[0])==0: return None, None
        d9 = []
        for r,c in zip(nines[0].tolist(), nines[1].tolist()):
            if r>45: continue
            for dr in range(-2,3):
                for dc in range(-2,3):
                    nr,nc=r+dr,c+dc
                    if 0<=nr<52 and 0<=nc<64 and int(g[nr,nc])==5:
                        d9.append((r,c)); break
                else: continue
                break
        if not d9: return None, None
        rn=min(r for r,c in d9); rx=max(r for r,c in d9)
        cn=min(c for r,c in d9); cx=max(c for r,c in d9)
        cr,cc=(rn+rx)//2,(cn+cx)//2
        pat=tuple(tuple(1 if int(g[cr+dr,cc+dc])==9 else 0 for dc in [-1,0,1]) for dr in [-1,0,1])
        return pat,(cr,cc)

    def _build_cs(self, arr):
        wall=(arr[:52]==4); cs=np.zeros((52,64),dtype=bool)
        for r in range(2,50):
            for c in range(2,62):
                if not np.any(wall[r-2:r+3,c-2:c+3]): cs[r,c]=True
        return cs

    def _sim(self, pos, d, cs):
        dr,dc={'U':(-1,0),'D':(1,0),'L':(0,-1),'R':(0,1)}[d]
        r,c=pos
        for _ in range(5):
            nr,nc=r+dr,c+dc
            if nr<2 or nr>=50 or nc<2 or nc>=62 or not cs[nr,nc]: break
            r,c=nr,nc
        return (r,c)

    def _bfs_to(self, cs, start, goal, max_depth=30):
        vis={start}; q=deque([(start,[])])
        best,bd=[],abs(start[0]-goal[0])+abs(start[1]-goal[1])
        while q:
            p,path=q.popleft()
            d=abs(p[0]-goal[0])+abs(p[1]-goal[1])
            if d<8: return path
            if d<bd: best,bd=path,d
            if len(path)>=max_depth: continue
            for di in 'UDLR':
                np2=self._sim(p,di,cs)
                if np2!=p and np2 not in vis: vis.add(np2); q.append((np2,path+[di]))
        return best

    def _keyboard_plan(self, arr):
        player=self._find_player(arr); rot=self._find_rotator(arr)
        dp,dc=self._find_door(arr); key=self._get_key(arr)
        if dc and player and abs(player[0]-dc[0])<6 and abs(player[1]-dc[1])<6:
            return [('D',None)]*3
        self.door_pat=dp
        if not all([player,rot,dp]): return self._generic_explore()
        cs=self._build_cs(arr)
        if dp==key:
            path=self._bfs_to(cs,player,dc)
            return [(d,None) for d in path]+[('U',None)]*3+[('D',None)]*3+[('L',None)]*3+[('R',None)]*3
        path_r=self._bfs_to(cs,player,rot); pos=player
        for m in path_r: pos=self._sim(pos,m,cs)
        path_d=self._bfs_to(cs,pos,dc)
        return [(d,None) for d in path_r+path_d]+[('U',None)]*3+[('D',None)]*3+[('L',None)]*3+[('R',None)]*3

    def _click_plan(self, arr):
        actions=[]
        b4=sorted(set((int(r),int(c)) for r,c in zip(*np.where(arr==4)) if r>=40))
        n9=sorted(set((int(r),int(c)) for r,c in zip(*np.where(arr==9)) if r<50))
        for r,c in b4: actions.append(('A6',{'x':c,'y':r}))
        for r,c in n9: actions.append(('A6',{'x':c,'y':r}))
        if not actions:
            for val in [9,5,4,11,8,10]:
                locs=np.where(arr==val)
                for r,c in zip(locs[0].tolist()[:20],locs[1].tolist()[:20]):
                    actions.append(('A6',{'x':int(c),'y':int(r)}))
                if len(actions)>50: break
        return actions if actions else [('A6',{'x':32,'y':32})]

    def _generic_explore(self):
        moves=[]
        for _ in range(3):
            moves.extend([('R',None)]*5+[('D',None)]*5+[('L',None)]*5+[('U',None)]*5)
        return moves

    def _mixed_plan(self, arr, frame):
        aa=frame.available_actions or []; plan=[]
        if any(a in aa for a in [1,2,3,4]): plan.extend(self._keyboard_plan(arr)[:20])
        if 6 in aa: plan.extend(self._click_plan(arr)[:30])
        if 5 in aa: plan.append(('5',None))
        return plan if plan else self._generic_explore()

    # ===== MAIN LOOP =====
    def is_done(self, frames, latest_frame):
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames, latest_frame):
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            self.phase='PLAN'; self.queue=[]; self.attempt=0
            return GameAction.RESET

        if latest_frame.levels_completed > self.levels:
            self.levels = latest_frame.levels_completed
            logger.info(f'LEVEL {self.levels} DONE!')
            self.phase='PLAN'; self.queue=[]; self.attempt=0; self.current_solution_idx=0

        # BFS solution available?
        if self.levels in self.bfs_solutions and not self.queue:
            sol = self.bfs_solutions[self.levels]
            logger.info(f'Using BFS solution for L{self.levels}: {len(sol)} actions')
            self.queue = list(sol)
            self.phase = 'BFS'

        if self.game_type is None:
            self.game_type = self._detect_type(latest_frame)
            self.available = latest_frame.available_actions or []
            logger.info(f'Game type: {self.game_type}, available: {self.available}')

        # Execute BFS queue
        if self.queue and self.phase == 'BFS':
            act_id, data = self.queue.pop(0)
            if data and 'x' in data:
                action = GameAction.ACTION6
                action.action_data.x = int(data['x'])
                action.action_data.y = int(data['y'])
                action.reasoning = f'v15 BFS click'
                return action
            am = {1:GameAction.ACTION1, 2:GameAction.ACTION2, 3:GameAction.ACTION3, 4:GameAction.ACTION4, 5:GameAction.ACTION5}
            action = am.get(act_id, GameAction.ACTION1)
            action.reasoning = f'v15 BFS step'
            return action

        # Online fallback
        arr = self._get_arr(latest_frame)
        if arr is None: return GameAction.RESET

        if self.game_type in ('keyboard','mixed') and self.door_pat:
            key = self._get_key(arr)
            if key == self.door_pat and self.phase == 'CROSSING':
                player = self._find_player(arr)
                _, dc = self._find_door(arr)
                if dc and player:
                    cs = self._build_cs(arr)
                    path = self._bfs_to(cs, player, dc)
                    self.queue = [(d,None) for d in path]+[('U',None)]*3+[('D',None)]*3+[('L',None)]*3+[('R',None)]*3
                    self.phase = 'TO_DOOR'

        if self.queue and self.phase != 'BFS':
            return self._execute_next()

        self.attempt += 1
        if self.attempt > 10: self.queue = self._generic_explore()
        elif self.game_type == 'keyboard': self.queue = self._keyboard_plan(arr); self.phase='CROSSING'
        elif self.game_type == 'click': self.queue = self._click_plan(arr); self.phase='CLICKING'
        elif self.game_type == 'mixed': self.queue = self._mixed_plan(arr, latest_frame); self.phase='MIXED'
        else: self.queue = self._generic_explore()

        if self.queue: return self._execute_next()
        return GameAction.ACTION1

    def _execute_next(self):
        act, data = self.queue.pop(0)
        if act == 'A6' and data:
            action = GameAction.ACTION6
            action.action_data.x = int(data['x'])
            action.action_data.y = int(data['y'])
            action.reasoning = f'v15 click ({data["y"]},{data["x"]})'
            return action
        elif act == '5':
            action = GameAction.ACTION5
            action.reasoning = 'v15 interact'
            return action
        am = {'U':GameAction.ACTION1,'D':GameAction.ACTION2,'L':GameAction.ACTION3,'R':GameAction.ACTION4}
        action = am.get(act, GameAction.ACTION1)
        action.reasoning = f'v15 {self.game_type} {self.phase}'
        return action

    def cleanup(self, *a, **kw):
        if self._cleanup: logger.info(f'RewindAgent v15 final: {self.levels}/7')
        super().cleanup(*a, **kw)

In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for gateway
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy repo
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Copy agent
    !cp /kaggle/working/rewind_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/rewind_agent.py

    # Patch __init__.py
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.rewind_agent import RewindAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    \"random\": Random,
    \"rewindagent\": RewindAgent,
}
""")

    # Write .env — point to environment_files for BFS
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=/kaggle/working/ARC-AGI-3-Agents/environment_files
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Run agent
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent rewindagent

In [ ]:
import pandas as pd

if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)